# Call Center Volume Forecasting
**Project 10 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **half-hourly call center arrival volumes** using the **Call Center Dataset** — a classic operations management dataset that demonstrates the challenges of forecasting high-frequency intraday series. Call centers are one of the original application domains for time series forecasting, and the problem remains operationally critical today.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting — Intraday & Weekly Panel |
| **Target variable** | `calls` (call arrivals per half-hour) |
| **Granularity** | 30-minute intervals → daily totals and weekly averages |
| **Frequency** | `30T` (30-minute), aggregated to `D` (daily) |
| **Primary TS library** | StatsForecast (AutoARIMA, AutoETS, AutoTheta) |
| **Dataset** | Cleveland Clinic call center / banking call center data (Ding, Koole & van der Mei) |
| **Kaggle dataset** | `jessemostipak/call-center-data` |

Call volumes follow a **double seasonality pattern**: intraday (half-hourly within day) and inter-week (day-of-week). This makes them a textbook example for SARIMA(m1)(m2) and models like `BATS`, `TBATS`, and `AutoARIMA` with multiple seasonal periods.

## Learning Objectives

1. **Understand double seasonality**: model both intraday (m=48 for 30-min intervals) and weekly (m=336) patterns simultaneously
2. **Aggregate appropriately** for different planning horizons: shift scheduling (hourly), weekly staffing (daily), annual budgeting (weekly)
3. **Apply StatsForecast `AutoARIMA`** with custom seasonal configurations
4. **Apply StatsForecast `AutoTheta`** — excellent for intraday data with multiple seasonal layers
5. **Erlang-C staffing formula**: translate call volume forecast into agent headcount requirement
6. **Evaluate with AHT (Average Handle Time)** context — forecasting calls is only useful when combined with handle time to size staffing
7. **Compare aggregation levels**: half-hourly vs. daily vs. weekly forecasting accuracy

## Problem Statement

Given N months of half-hourly call arrival data, **forecast the next 4 weeks** of daily call volumes to support:
1. **Shift scheduling**: agent schedules published 2 weeks in advance
2. **Workforce management (WFM)**: long-term hiring/training decisions
3. **SLA planning**: ensuring staffing meets ≥80% service level (calls answered within 20 seconds)

Accurate call volume forecasts directly translate into cost savings: understaffing leads to SLA breaches (contract penalties), overstaffing to idle agents (wasted salary cost).

## Why This Project Matters

- **Workforce management at scale**: call centers employ 3+ million people in the US alone; 5% better forecast accuracy = millions in reduced overstaffing costs
- **Real-time re-forecasting**: modern WFM systems re-forecast every 15 minutes based on actual call arrivals vs. predicted
- **SLA-driven staffing**: the Erlang-C formula converts a call volume forecast + AHT + target service level into an agent count — forecasting accuracy directly determines whether SLAs are met
- **Generalises to other queuing systems**: airline check-in, hospital ED arrivals, retail store traffic, IT help desk tickets

## Dataset Overview

The **Call Center Data** dataset contains records of inbound calls to a medium-sized financial institution:
- Daily call volumes over 1+ years
- Half-hourly breakdown within each day
- Some datasets also include: call type (billing, technical, sales), average handle time (AHT), and service level

If the Kaggle dataset is unavailable, we generate a synthetic dataset with realistic call center patterns (double seasonality, lunch dip, morning ramp-up, end-of-month spikes).

### Kaggle fallback dataset options
- `jessemostipak/call-center-data`
- `dharmarajsinh/call-center-dataset`
- `vjckling/customerservicedata`

## Dataset Source & License Notes

- **Primary**: Call Center Data — financial institution (academic datasets, freely used for teaching)
- **Kaggle**: Multiple call center datasets available; see alternatives above
- **License**: Educational use; synthetic generation used when real data unavailable

## Environment Setup

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pip in {
    "kagglehub": "kagglehub", "pandas": "pandas", "numpy": "numpy",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "plotly": "plotly",
    "statsforecast": "statsforecast", "statsmodels": "statsmodels",
    "scikit_learn": "scikit-learn", "lazypredict": "lazypredict",
    "flaml": "flaml",
}.items():
    try: __import__(imp)
    except ImportError: _pip(pip)
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoTheta, SeasonalNaive,
    Naive, WindowAverage
)

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration & Constants

In [ ]:
PROJECT_NAME  = "Call Center Volume Forecasting"
KAGGLE_SLUG   = "jessemostipak/call-center-data"   # Primary attempt
KAGGLE_ALT    = "dharmarajsinh/call-center-dataset" # Fallback
TARGET_COL    = "calls"
FREQ_DAILY    = "D"
FREQ_WEEKLY   = "W"
HORIZON       = 28               # 4 weeks = 28 days ahead (daily model)
SEASON_LEN_W  = 7               # Weekly seasonality (7 days)
SEASON_LEN_ANN = 52             # Annual seasonality (52 weeks)
TEST_DAYS     = 28
FLAML_BUDGET  = 90
RANDOM_STATE  = 42

AHT_SECONDS   = 300             # Average Handle Time = 5 minutes
TARGET_SL     = 0.80            # Service Level: 80% calls answered in 20 seconds
SL_THRESHOLD  = 20              # Seconds

print(f"Project : {PROJECT_NAME}")
print(f"AHT     : {AHT_SECONDS}s | Target SL: {TARGET_SL*100:.0f}% in {SL_THRESHOLD}s")
print(f"Horizon : {HORIZON} days | Test: {TEST_DAYS} days")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    if os.environ.get(k): print(f"✓ {k} set."); kaggle_ok = True; break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists(): print("✓ kaggle.json found."); kaggle_ok = True
if not kaggle_ok:
    print("NO KAGGLE CREDENTIALS. Will use synthetic data.")
else:
    print("Credentials OK.")

## Dataset Download (with Automatic Fallback to Synthetic)

In [ ]:
import kagglehub

call_df = None

for slug in [KAGGLE_SLUG, KAGGLE_ALT]:
    try:
        data_path = pathlib.Path(kagglehub.dataset_download(slug))
        csv_files = list(data_path.rglob("*.csv"))
        if csv_files:
            call_df = pd.read_csv(csv_files[0])
            print(f"Loaded from: {slug}")
            print(f"Shape: {call_df.shape}")
            print(call_df.head(3).to_string())
            break
    except Exception as e:
        print(f"{slug} failed: {e}")

if call_df is None:
    print("\n=== USING SYNTHETIC CALL CENTER DATA ===")
    print("Both Kaggle datasets unavailable. Generating realistic synthetic data.\n")

## Synthetic Call Center Data Generation

In [ ]:
np.random.seed(RANDOM_STATE)
print("Generating 2 years of synthetic half-hourly call center data...")

START     = pd.Timestamp("2022-01-01")
N_DAYS    = 730
INTERVALS = 48   # 30-min intervals per day

# Half-hour of day load factors (48 intervals, 0=midnight, 17=8:30AM peak)
# Typical financial services call center pattern:
intraday_factors = np.array([
    0.02,0.02,0.02,0.01,0.01,0.01,  # 00:00-02:30
    0.01,0.01,0.02,0.03,0.04,0.06,  # 03:00-05:30
    0.08,0.14,0.18,0.22,0.28,0.38,  # 06:00-08:30
    0.50,0.75,0.90,1.00,0.95,0.85,  # 09:00-11:30     ← morning peak
    0.75,0.65,0.82,0.88,0.90,0.88,  # 12:00-14:30
    0.82,0.78,0.70,0.65,0.58,0.55,  # 15:00-17:30     ← early afternoon decline
    0.42,0.35,0.28,0.20,0.15,0.10,  # 18:00-20:30
    0.08,0.06,0.04,0.03,0.02,0.02,  # 21:00-23:30
])
assert len(intraday_factors) == 48

dow_factors = np.array([1.10,1.15,1.12,1.08,1.05,0.40,0.20])  # Mon-Sun (weekend ~20-40% weekday)
month_factors = {1:0.95,2:0.90,3:1.00,4:1.02,5:1.05,6:1.03,
                  7:0.95,8:0.93,9:1.03,10:1.05,11:1.00,12:1.08}
BASE_CALLS_PER_INTERVAL = 180  # ~8,640 calls/day on a Monday

records = []
for day_i in range(N_DAYS):
    dt_day = START + pd.Timedelta(days=day_i)
    dow_f  = dow_factors[dt_day.dayofweek]
    mon_f  = month_factors[dt_day.month]
    trend_f = 1.0 + 0.05 * (day_i / N_DAYS)  # slow 5% growth over 2 years
    
    # End-of-month billing spike (+20% on the 1st-3rd of each month)
    eom_f = 1.20 if dt_day.day in [1, 2, 3] else 1.0
    
    for interval_i in range(INTERVALS):
        dt_ts  = dt_day + pd.Timedelta(minutes=30 * interval_i)
        intra_f = intraday_factors[interval_i]
        noise_f = 1.0 + np.random.normal(0, 0.12)
        calls = max(0, int(BASE_CALLS_PER_INTERVAL * intra_f * dow_f * mon_f * trend_f * eom_f * noise_f))
        records.append({"ds": dt_ts, "calls": calls})

raw_df = pd.DataFrame(records)
print(f"Generated: {len(raw_df):,} half-hourly records")
print(f"Date range: {raw_df['ds'].min()} to {raw_df['ds'].max()}")
print()

# Aggregate to daily
daily_calls = raw_df.groupby(raw_df["ds"].dt.normalize())["calls"].sum().reset_index()
daily_calls.columns = ["date","calls"]
daily_calls["unique_id"] = "call_center_main"
daily_calls = daily_calls.rename(columns={"date":"ds","calls":"y"})

print(f"Daily totals: {len(daily_calls)} days")
print(f"Daily calls stats: mean={daily_calls['y'].mean():,.0f}, std={daily_calls['y'].std():,.0f}")
print()
print(daily_calls.head(7).to_string(index=False))

## Data Validation Checks

In [ ]:
print("=" * 55)
print("DATA VALIDATION — Call Center")
print("=" * 55)

print(f"\nDaily series: {len(daily_calls)} days")
print(f"Missing: {daily_calls['y'].isnull().sum()}")
print(f"Zeros: {(daily_calls['y']==0).sum()}")
print(f"Weekends present: {daily_calls['ds'].dt.dayofweek.isin([5,6]).sum()} days")

print(f"\nCalls per day statistics:")
print(daily_calls['y'].describe().round(0).to_string())

# Validate weekend vs weekday distribution
daily_calls["is_weekend"] = daily_calls["ds"].dt.dayofweek >= 5
wd_stats = daily_calls.groupby("is_weekend")["y"].mean()
print(f"\nWeekday avg calls: {wd_stats[False]:,.0f}")
print(f"Weekend avg calls : {wd_stats[True]:,.0f}")
print(f"Weekday/Weekend ratio: {wd_stats[False]/wd_stats[True]:.1f}x")

print(f"\nEnd-of-month spike validation:")
daily_calls["day_of_month"] = daily_calls["ds"].dt.day
eom = daily_calls.groupby("day_of_month")["y"].mean()
print(f"Day 1 avg: {eom[1]:,.0f}  |  Day 15 avg: {eom[15]:,.0f}  |  Day 30 avg: {eom.get(30, eom.max()):,.0f}")

## Exploratory Data Analysis

In [ ]:
fig = px.line(daily_calls, x="ds", y="y",
              title="Daily Call Volume — Full 2-Year History",
              labels={"ds":"Date","y":"Calls per Day"},
              template="plotly_white")
fig.show()

In [ ]:
# ── Seasonal decomposition ────────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose
y_series = daily_calls.set_index("ds")["y"]
dec = seasonal_decompose(y_series, model="additive", period=7)
dec.plot()
plt.suptitle("Daily Call Volume Decomposition (Weekly Seasonality, period=7)")
plt.tight_layout(); plt.show()

In [ ]:
# ── Intraday pattern ─────────────────────────────────────────────────
raw_df["interval"] = raw_df["ds"].dt.hour * 2 + (raw_df["ds"].dt.minute // 30)
raw_df["dow"]      = raw_df["ds"].dt.day_name()

intraday_avg = (raw_df[~raw_df["ds"].dt.dayofweek.isin([5,6])]
                .groupby("interval")["calls"].mean())

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(intraday_avg.index / 2, intraday_avg.values, marker="o", markersize=3, lw=2, color="#2563EB")
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)], rotation=30)
ax.set_title("Average Intraday Call Pattern (Weekdays Only)")
ax.set_xlabel("Time of Day")
ax.set_ylabel("Avg Calls per 30 Minutes")
plt.tight_layout(); plt.show()
print(f"Peak interval: {intraday_avg.idxmax()/2:.1f}:00 ({intraday_avg.max():.0f} avg calls)")

In [ ]:
# ── Day of week pattern ────────────────────────────────────────────────
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
daily_calls["dow"] = daily_calls["ds"].dt.day_name()
dow_avg = daily_calls.groupby("dow")["y"].mean().reindex(dow_order)
fig = px.bar(x=dow_avg.index, y=dow_avg.values,
             title="Average Daily Calls by Day of Week (Double Seasonality Visible)",
             labels={"x":"Day","y":"Avg Calls"},
             template="plotly_white",
             color=dow_avg.values, color_continuous_scale="Blues")
fig.show()

## Target Analysis

In [ ]:
y = daily_calls["y"]
print("Daily call volume statistics:")
print(f"  Mean   : {y.mean():,.0f}")
print(f"  Std    : {y.std():,.0f}")
print(f"  CV     : {y.std()/y.mean()*100:.1f}%")
print(f"  Min    : {y.min():,.0f}  (weekend)")
print(f"  Max    : {y.max():,.0f}")
print()

from pandas.plotting import autocorrelation_plot
fig, ax = plt.subplots(figsize=(14, 4))
autocorrelation_plot(y, ax=ax)
ax.set_title("Autocorrelation — Daily Call Volume (Note lag-7 spike = weekly seasonality)")
ax.set_xlim(0, min(60, len(y)//2))
ax.axvline(7, color="red", lw=0.8, linestyle="--", label="lag 7")
ax.axvline(14, color="orange", lw=0.8, linestyle="--", label="lag 14")
ax.legend()
plt.tight_layout(); plt.show()

for lag in [1, 2, 5, 7, 14, 21, 28, 365]:
    if lag < len(y)//2:
        print(f"  ACF lag {lag:>4}: {y.autocorr(lag):.3f}")

## Train / Validation / Test Split

In [ ]:
n = len(daily_calls)
test_idx  = slice(n - TEST_DAYS, n)
train_idx = slice(0, n - TEST_DAYS)

df_train = daily_calls.iloc[train_idx].copy()
df_test  = daily_calls.iloc[test_idx].copy()
df_train_noid = df_train[["ds","y"]].copy()

print(f"Train: {len(df_train)} days  ({df_train['ds'].min().date()} → {df_train['ds'].max().date()})")
print(f"Test : {len(df_test)} days   ({df_test['ds'].min().date()} → {df_test['ds'].max().date()})")
actual_test = df_test["y"].values

## Feature Engineering for Tabular Models

In [ ]:
def make_lag_features(df_d):
    out = df_d.copy().reset_index(drop=True)
    for lag in [1, 2, 3, 7, 14, 21, 28]:
        out[f"lag_{lag}d"] = out["y"].shift(lag)
    out["roll_7d"]  = out["y"].shift(1).rolling(7).mean()
    out["roll_28d"] = out["y"].shift(1).rolling(28).mean()
    out["dow"]      = out["ds"].dt.dayofweek
    out["month"]    = out["ds"].dt.month
    out["day_of_month"] = out["ds"].dt.day
    out["is_weekend"]   = (out["dow"] >= 5).astype(int)
    out["is_eom_spike"] = out["day_of_month"].isin([1,2,3]).astype(int)
    return out

feat_all = make_lag_features(daily_calls)
FEAT_COLS = [c for c in feat_all.columns if c not in ("ds","unique_id","y","dow") and feat_all[c].dtype != object] + ["dow"]
feat_tr = feat_all.iloc[train_idx].dropna()
feat_te = feat_all.iloc[test_idx].dropna()
print(f"Features ({len(FEAT_COLS)}): {FEAT_COLS}")
print(f"Train: {len(feat_tr)} | Test: {len(feat_te)}")

## Baselines

In [ ]:
def evaluate(actual, predicted, name):
    a = np.array(actual, float).flatten()
    p = np.array(predicted, float).flatten()
    n = min(len(a), len(p))
    mae_v  = mean_absolute_error(a[:n], p[:n])
    rmse_v = np.sqrt(mean_squared_error(a[:n], p[:n]))
    mape_v = np.mean(np.abs((a[:n] - p[:n]) / np.maximum(a[:n], 1))) * 100
    print(f"  {name:<40s}  MAE:{mae_v:>7.0f}  RMSE:{rmse_v:>7.0f}  MAPE:{mape_v:>5.1f}%")
    return {"model": name, "MAE": mae_v, "RMSE": rmse_v, "MAPE": mape_v}

results = []
y_tr = df_train["y"].values
print("BASELINES (daily call volume, test = last 28 days):")
# Seasonal naive (same weekday, 1 week ago)
sn7_pred = np.array([y_tr[-(7 - (i % 7))] for i in range(TEST_DAYS)])
results.append(evaluate(actual_test, sn7_pred, "Seasonal Naive (7-day)"))
# 4-week moving average
results.append(evaluate(actual_test, np.full(TEST_DAYS, y_tr[-28:].mean()), "28-Day Moving Average"))
# Last same-weekday value
dow_medians = pd.Series(y_tr, index=df_train["ds"]).groupby(lambda x: x.dayofweek).median()
swd_pred = np.array([dow_medians[df_test.iloc[i]["ds"].dayofweek] for i in range(TEST_DAYS)])
results.append(evaluate(actual_test, swd_pred, "Day-of-Week Median"))

## LazyPredict Benchmark

In [ ]:
if len(feat_tr) >= 10 and len(feat_te) >= 1:
    try:
        lr = LazyRegressor(verbose=0, ignore_warnings=True, predictions=True)
        lz_m, lz_p = lr.fit(feat_tr[FEAT_COLS], feat_te[FEAT_COLS],
                             feat_tr["y"], feat_te["y"])
        print("LazyPredict top 10 (daily test):")
        print(lz_m.head(10).to_string())
        best_lz = lz_m.index[0]
        results.append(evaluate(actual_test[:len(lz_p[best_lz])], lz_p[best_lz], f"LazyPredict-{best_lz}"))
    except Exception as e:
        print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
X_tv = feat_tr[FEAT_COLS]; y_tv = feat_tr["y"]
flaml_m = AutoML()
flaml_m.fit(X_tv, y_tv, task="regression", time_budget=FLAML_BUDGET,
            metric="mae", verbose=0, seed=RANDOM_STATE)
if len(feat_te) > 0:
    fp = flaml_m.predict(feat_te[FEAT_COLS])
    results.append(evaluate(actual_test[:len(fp)], fp, f"FLAML ({flaml_m.best_estimator})"))
    print(f"FLAML best: {flaml_m.best_estimator}")

## StatsForecast — AutoARIMA, AutoETS, AutoTheta

In [ ]:
sf_train = df_train[["unique_id","ds","y"]].copy()

sf = StatsForecast(
    models=[
        AutoARIMA(season_length=7, alias="AutoARIMA_7"),
        AutoETS(season_length=7, alias="AutoETS_7"),
        AutoTheta(season_length=7, alias="AutoTheta_7"),
        SeasonalNaive(season_length=7, alias="SeasonalNaive_7"),
        WindowAverage(window_size=7, alias="WindowAvg7"),
    ],
    freq=FREQ_DAILY,
    n_jobs=1,
)
print("Fitting StatsForecast (daily, season=7)...")
sf.fit(sf_train)
sf_fcst = sf.predict(h=TEST_DAYS, level=[80, 95])
print(sf_fcst.head(5).to_string())

In [ ]:
print("StatsForecast results:")
for col in ["AutoARIMA_7","AutoETS_7","AutoTheta_7","SeasonalNaive_7","WindowAvg7"]:
    if col in sf_fcst.columns:
        results.append(evaluate(actual_test, sf_fcst[col].values, f"StatsForecast-{col}"))

In [ ]:
# ── Plot forecast ─────────────────────────────────────────────────────
plot_start = df_train.iloc[-60:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_start["ds"], y=plot_start["y"],
                          name="Train (last 60d)", mode="lines", line=dict(color="#2563EB")))
fig.add_trace(go.Scatter(x=df_test["ds"], y=df_test["y"],
                          name="Actual (test 28d)", mode="lines+markers",
                          line=dict(color="#1E3A5F", dash="dot")))
for col, clr in [("AutoARIMA_7","#EF4444"),("AutoETS_7","#F59E0B"),("AutoTheta_7","#10B981")]:
    if col in sf_fcst.columns:
        fig.add_trace(go.Scatter(x=sf_fcst["ds"], y=sf_fcst[col],
                                  name=col, mode="lines",
                                  line=dict(dash="dash", color=clr)))
# Add PI band for AutoARIMA
if "AutoARIMA_7-lo-80" in sf_fcst.columns:
    fig.add_trace(go.Scatter(
        x=list(sf_fcst["ds"]) + list(sf_fcst["ds"])[::-1],
        y=list(sf_fcst["AutoARIMA_7-hi-80"]) + list(sf_fcst["AutoARIMA_7-lo-80"])[::-1],
        fill="toself", fillcolor="rgba(239,68,68,0.1)", line=dict(color="rgba(255,255,255,0)"),
        name="AutoARIMA 80% PI"
    ))
fig.update_layout(title="Daily Call Volume Forecast — StatsForecast vs Actual",
                  xaxis_title="Date", yaxis_title="Daily Calls",
                  template="plotly_white")
fig.show()

## Erlang-C Staffing Model

In [ ]:
# Erlang-C: translate forecast call volume → required agents
# Simplified implementation for demonstration

def erlang_c_agents(calls_per_hour, aht_s, target_sl, sl_threshold_s):
    # Estimate required agents using Erlang-C formula.
    # calls_per_hour: expected arrival rate (calls/hour)
    # aht_s: average handle time (seconds)
    # target_sl: target service level e.g. 0.80
    # sl_threshold_s: SL threshold in seconds e.g. 20
    import math
    mu = 3600 / aht_s  # service rate (calls per hour per agent)
    if calls_per_hour <= 0:
        return 1
    # Traffic intensity A = lambda / mu
    A = calls_per_hour / mu
    if A <= 0:
        return 1
    # Minimum agents = ceil(A) + 1
    N_min = math.ceil(A) + 1
    # Iterate up to find N that achieves target SL
    for N in range(N_min, N_min + 30):
        # Erlang C approximation (numerically stable version)
        rho = A / N
        if rho >= 1: continue
        # P(wait) approximation
        C_num = (A**N / math.factorial(N)) * (N / (N - A))
        denom = sum(A**k / math.factorial(k) for k in range(N)) + C_num
        if denom == 0: continue
        C = C_num / denom  # Probability of waiting (Erlang C)
        # Service level = 1 - C * e^(-(N - A) * (sl_threshold_s / aht_s))
        sl = 1 - C * np.exp(-(N - A) * (sl_threshold_s * mu / N))
        if sl >= target_sl:
            return N
    return N_min + 30

# Apply to best StatsForecast forecast
best_sf_col = None
for col in ["AutoARIMA_7","AutoETS_7","AutoTheta_7"]:
    if col in sf_fcst.columns:
        best_sf_col = col; break

if best_sf_col:
    # Convert daily → peak hourly (use 10AM = 2× daily average as peak hour proxy)
    PEAK_HOUR_MULTIPLIER = 2.0
    daily_to_hourly = PEAK_HOUR_MULTIPLIER / 10  # ~10 active hours per day
    
    print(f"Erlang-C Staffing Requirement — Based on {best_sf_col} forecast:")
    print(f"AHT={AHT_SECONDS}s, SL target={TARGET_SL*100:.0f}% in {SL_THRESHOLD}s\n")
    print(f"{'Date':<12} {'Forecast Calls':>15} {'Peak CPH':>10} {'Agents Needed':>15}")
    print("-" * 55)
    for _, row in sf_fcst.iterrows():
        daily_fcst = max(0, row[best_sf_col])
        peak_cph   = daily_fcst * daily_to_hourly
        agents     = erlang_c_agents(peak_cph, AHT_SECONDS, TARGET_SL, SL_THRESHOLD)
        print(f"{str(row['ds'].date()):<12} {daily_fcst:>15,.0f} {peak_cph:>10,.1f} {agents:>15}")

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
print("=" * 72)
print("ALL MODELS — ranked by MAE (daily call volume forecasting)")
print("=" * 72)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\n{'TOP 3':^72}")
print("=" * 72)
print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y=["MAE","RMSE"], barmode="group",
             title="Model Comparison — MAE and RMSE (Daily Calls)",
             template="plotly_white")
fig.update_layout(xaxis_tickangle=-40)
fig.show()

## Error Analysis with Day-of-Week Breakdown

In [ ]:
best_name = results_df.iloc[0]["model"]
for col in ["AutoARIMA_7","AutoETS_7","AutoTheta_7"]:
    if col in best_name and col in sf_fcst.columns:
        pred = sf_fcst[col].values
        errors = actual_test[:len(pred)] - pred
        df_test_copy = df_test.copy().reset_index(drop=True)
        df_test_copy["error"] = errors[:len(df_test_copy)]
        df_test_copy["dow_name"] = df_test_copy["ds"].dt.day_name()
        
        print(f"Error by day of week ({col}):")
        doe_by_dow = df_test_copy.groupby("dow_name")["error"].agg(["mean","std"]).round(0)
        print(doe_by_dow.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).to_string())
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].bar(range(len(errors)), errors,
                    color=["#EF4444" if e<0 else "#10B981" for e in errors])
        axes[0].axhline(0, color="black", lw=1)
        axes[0].set_title(f"{col}: Daily Error (actual − predicted)")
        axes[0].set_xlabel("Day")
        axes[0].set_ylabel("Calls Error")
        
        axes[1].scatter(actual_test[:len(pred)]/1000, pred/1000, alpha=0.8, color="#2563EB")
        lim = max(actual_test[:len(pred)].max(), pred.max()) / 1000
        axes[1].plot([0,lim],[0,lim],"r--"); axes[1].set_aspect("equal")
        axes[1].set_title("Actual vs Predicted (000s)")
        axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")
        plt.tight_layout(); plt.show()
        break

## Interpretation & Insights

### Double Seasonality in Call Center Data
Call center arrivals exhibit two distinct seasonal patterns simultaneously:
1. **Intraday pattern** (period=48 for 30-min intervals): ramp-up from 8 AM, peak 9-11 AM, lunch dip, afternoon secondary peak, rapid drop after 6 PM
2. **Weekly pattern** (period=7 for daily data): Monday–Friday high, Saturday reduced, Sunday very low

`AutoARIMA(season_length=7)` captures the weekly pattern well. For modelling the full intraday pattern, use `season_length=48` on the raw half-hourly data.

### Staffing Implications
The Erlang-C model translates the call volume forecast into agent requirements. Key insight:
- A 10% over-forecast → 1-2 extra agents scheduled unnecessarily per shift
- A 10% under-forecast → SLA breach (< 80% calls answered in 20 seconds) → contract penalties
- Weekend forecasting is critical because absolute volume is lower but *relative errors are larger*

### End-of-Month Pattern
The first 3 days of each month show consistent billing inquiry spikes (+20-25%). This is captured in the `is_eom_spike` feature for tabular models and automatically captured by the strong lag-1 and lag-7 autocorrelation in ARIMA models.

## Limitations

1. **No inter-call arrival model**: call arrivals are non-stationary Poisson processes; SARIMA treats them as normally distributed — OK for daily aggregates, wrong for sub-minute intervals
2. **Constant AHT assumption**: real call centers have AHT variation by call type, agent experience, and time of day
3. **Missing call type segmentation**: billing, technical, and sales calls have different peak times and AHT distributions
4. **No agent shrinkage factor**: Erlang-C gives minimum agents; add 30% shrinkage for breaks, training, and schedule adherence
5. **Holiday handling**: the model doesn't have explicit holiday calendars; statutory holidays cause sudden demand drops

## How to Improve This Project

1. **Half-hourly model with SARIMA(48)**: use raw 30-min data with `freq="30T"` and `season_length=48` — captures intraday peak more precisely
2. **Dual seasonality model**: use `MSTL` (Multiple Seasonal-Trend Decomposition) from `statsforecast` for simultaneous 7-day + 365-day seasonality
3. **Call type panel**: segment by inbound call type and model as a multi-series panel — billing peaks on the 1st, sales peaks on Mondays
4. **Real-time adjustment**: use actual calls-so-far today as an intercept correction for the remaining hours of the same day
5. **Erlang-C extension**: add agent shrinkage (breaks, training, absenteeism = 30% overhead) and oncall buffer

## Production Considerations

1. **WFM integration**: call volume forecasts feed directly into Verint, NICE, or Genesys WFM systems via scheduled API calls
2. **Intraday reforecast**: at the half-hourly mark, re-estimate today's expected call volume based on actual arrivals so far
3. **Alert thresholds**: if actual/forecast > 1.15 for 2 consecutive hours, trigger on-call escalation to bring in part-time agents
4. **Forecast lock window**: freeze the 2-week-ahead forecast every Monday 6 AM for publishing agent schedules — changes after that require manager approval
5. **SLA breach simulation**: run Monte Carlo on the Erlang-C model with forecast uncertainty to estimate SLA breach probability

## Common Mistakes to Avoid

1. **Not separating weekday and weekend models**: treating all days identically produces poor weekend forecasts due to the dramatically different volume profile
2. **Forgetting end-of-month spikes**: billing inquiry surges on the 1st–3rd of each month are predictable and should be explicitly modelled
3. **Treating daily averages as uniform**: 180 calls/day does not mean 7.5 calls/hour — the intraday distribution is extremely peaked
4. **Using MAPE on Sundays**: near-zero Sunday volumes → infinite MAPE; always use MAE or RMSE for call center evaluation
5. **Skipping the Erlang-C step**: call volume forecasting is only useful if translated into agent headcount — without a staffing model, the forecast has no operational meaning

## Mini Challenge / Exercises

1. **Half-hourly SARIMA**: aggregate to 30-min intervals and fit `AutoARIMA(season_length=48)` — how does intraday MAPE compare to daily MAPE?
2. **Holiday adjustment**: add a `is_holiday` feature for major US federal holidays (New Year, Independence Day, Thanksgiving, Christmas) — measure MAPE improvement
3. **Erlang-C sensitivity analysis**: vary AHT from 200s to 400s and show how required agents changes for the same forecast
4. **Per-weekday models**: fit 5 separate ARIMA models (one per weekday) and ensemble them — does specialisation improve accuracy?
5. **Service level simulation**: using your best forecast's 80% prediction interval, simulate the probability of SLA breach if you schedule for the point forecast vs. the 90th percentile

## Final Summary & Key Takeaways

### What We Did
- Generated 2 years of synthetic half-hourly call center data with realistic intraday + weekly + end-of-month patterns
- Validated data quality: weekend/weekday ratio, intraday profile, autocorrelation structure
- Aggregated to daily totals and analysed seasonal decomposition
- Built lag-feature baselines with Seasonal Naive, Day-of-Week Median, and Moving Average
- Benchmarked LazyPredict and FLAML AutoML on tabular lag features
- Fitted StatsForecast AutoARIMA, AutoETS, and AutoTheta with `season_length=7`
- Translated calls forecast into agent requirements using the Erlang-C staffing formula
- Analysed per-day-of-week error patterns to identify model blind spots

### Key Takeaways
1. **Weekly seasonality (period=7) is the primary driver** of call center forecast error — get this right first
2. **The Erlang-C model bridges forecast and operations** — call volume is only useful when converted to agents
3. **End-of-month spikes are predictable** — encode `day_of_month ≤ 3` as a feature for systematic improvement
4. **StatsForecast AutoTheta** often outperforms AutoARIMA on weekly-seasonal count series
5. **Double seasonality** (intraday + weekly) requires either multi-period SARIMA or decomposition approaches for high-frequency data

---
*Notebook #10 of 50 — Time Series Forecasting Portfolio*
*Dataset: Synthetic Call Center (double seasonality) | Library: StatsForecast | Freq: Daily*